# 📱 Deteksi Kecanduan Smartphone Menggunakan Ensemble Machine Learning

**Jurnal:** Sinta 3  
**Metode:** Gradient Boosting, Random Forest, SVM, Logistic Regression  
**Evaluasi:** Accuracy, Precision, Recall, F1-Score, AUC-ROC, Confusion Matrix  
**Optimasi:** GridSearchCV dengan Stratified K-Fold Cross Validation  

---

## Abstrak
Penelitian ini mengusulkan sistem deteksi kecanduan smartphone menggunakan pendekatan ensemble machine learning. Dataset terdiri dari 7.500 data pengguna dengan 13 fitur perilaku penggunaan smartphone. Eksperimen membandingkan empat algoritma klasifikasi dengan optimasi hyperparameter menggunakan GridSearchCV dan evaluasi menggunakan Stratified 10-Fold Cross Validation.

## 1. Import Library

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier, AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score,
    GridSearchCV, learning_curve
)
from sklearn.preprocessing import LabelEncoder, StandardScaler, label_binarize
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    roc_curve, auc, ConfusionMatrixDisplay
)
from sklearn.utils import resample
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance

# Seed untuk reprodusibilitas
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("✅ Semua library berhasil diimport")

## 2. Load dan Eksplorasi Data (EDA)

In [ ]:
# Load dataset
df = pd.read_csv('Smartphone_Usage_And_Addiction_Analysis_7500_Rows.csv')

print("=" * 60)
print(f"📊 INFORMASI DATASET")
print("=" * 60)
print(f"Jumlah baris   : {df.shape[0]:,}")
print(f"Jumlah kolom   : {df.shape[1]}")
print(f"\nKolom:")
for col in df.columns:
    print(f"  - {col} ({df[col].dtype})")

print(f"\n🎯 Distribusi Target (addicted_label):")
vc = df['addicted_label'].value_counts()
for label, count in vc.items():
    pct = count / len(df) * 100
    label_str = 'Kecanduan' if label == 1 else 'Tidak Kecanduan'
    print(f"  {label_str} ({label}): {count:,} ({pct:.1f}%)")

In [ ]:
# Tampilkan 5 baris pertama
print("📋 5 Baris Pertama Dataset:")
df.head()

In [ ]:
# Statistik deskriptif
print("📈 Statistik Deskriptif Fitur Numerik:")
df.describe().round(3)

In [ ]:
# Cek missing values
print("🔍 Missing Values per Kolom:")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Jumlah': missing, 'Persentase (%)': missing_pct})
print(missing_df[missing_df['Jumlah'] > 0])
print(f"\nTotal missing values: {missing.sum()}")

In [ ]:
# Visualisasi EDA - Figure 1: Distribusi Kelas dan Fitur Utama
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Gambar 1. Eksplorasi Data Awal (EDA)', fontsize=14, fontweight='bold')

# 1. Distribusi target
colors_target = ['#2196F3', '#F44336']
labels_target = ['Tidak Kecanduan', 'Kecanduan']
counts_target = df['addicted_label'].value_counts().sort_index()
axes[0,0].pie(counts_target, labels=labels_target, colors=colors_target,
              autopct='%1.1f%%', startangle=90, textprops={'fontsize': 11})
axes[0,0].set_title('Distribusi Label Kecanduan', fontweight='bold')

# 2. Distribusi screen time
axes[0,1].hist(df[df['addicted_label']==0]['daily_screen_time_hours'],
               alpha=0.6, color='#2196F3', label='Tidak Kecanduan', bins=25, edgecolor='white')
axes[0,1].hist(df[df['addicted_label']==1]['daily_screen_time_hours'],
               alpha=0.6, color='#F44336', label='Kecanduan', bins=25, edgecolor='white')
axes[0,1].set_xlabel('Durasi Layar Harian (Jam)')
axes[0,1].set_ylabel('Frekuensi')
axes[0,1].set_title('Distribusi Screen Time Harian', fontweight='bold')
axes[0,1].legend()

# 3. Distribusi jam tidur
axes[0,2].hist(df[df['addicted_label']==0]['sleep_hours'],
               alpha=0.6, color='#2196F3', label='Tidak Kecanduan', bins=25, edgecolor='white')
axes[0,2].hist(df[df['addicted_label']==1]['sleep_hours'],
               alpha=0.6, color='#F44336', label='Kecanduan', bins=25, edgecolor='white')
axes[0,2].set_xlabel('Jam Tidur')
axes[0,2].set_ylabel('Frekuensi')
axes[0,2].set_title('Distribusi Jam Tidur', fontweight='bold')
axes[0,2].legend()

# 4. Distribusi gender
gender_counts = df.groupby(['gender', 'addicted_label']).size().unstack(fill_value=0)
gender_counts.plot(kind='bar', ax=axes[1,0], color=['#2196F3', '#F44336'],
                   edgecolor='white', width=0.7)
axes[1,0].set_xlabel('Gender')
axes[1,0].set_ylabel('Jumlah')
axes[1,0].set_title('Distribusi Gender vs Label', fontweight='bold')
axes[1,0].set_xticklabels(axes[1,0].get_xticklabels(), rotation=0)
axes[1,0].legend(['Tidak Kecanduan', 'Kecanduan'])

# 5. Distribusi stress level
stress_counts = df.groupby(['stress_level', 'addicted_label']).size().unstack(fill_value=0)
stress_counts.plot(kind='bar', ax=axes[1,1], color=['#2196F3', '#F44336'],
                   edgecolor='white', width=0.7)
axes[1,1].set_xlabel('Tingkat Stres')
axes[1,1].set_ylabel('Jumlah')
axes[1,1].set_title('Stress Level vs Label', fontweight='bold')
axes[1,1].set_xticklabels(axes[1,1].get_xticklabels(), rotation=0)
axes[1,1].legend(['Tidak Kecanduan', 'Kecanduan'])

# 6. Distribusi usia
axes[1,2].hist(df[df['addicted_label']==0]['age'],
               alpha=0.6, color='#2196F3', label='Tidak Kecanduan', bins=18, edgecolor='white')
axes[1,2].hist(df[df['addicted_label']==1]['age'],
               alpha=0.6, color='#F44336', label='Kecanduan', bins=18, edgecolor='white')
axes[1,2].set_xlabel('Usia')
axes[1,2].set_ylabel('Frekuensi')
axes[1,2].set_title('Distribusi Usia', fontweight='bold')
axes[1,2].legend()

plt.tight_layout()
plt.savefig('gambar1_eda.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Gambar 1 tersimpan: gambar1_eda.png")

## 3. Preprocessing Data

In [ ]:
# Salin dataframe
df_processed = df.copy()

# Drop kolom ID dan target alternatif (addiction_level tidak digunakan)
cols_to_drop = ['transaction_id', 'user_id', 'addiction_level']
df_processed = df_processed.drop(columns=cols_to_drop)
print(f"✅ Drop kolom: {cols_to_drop}")

# Encoding variabel kategorik
le = LabelEncoder()
categorical_cols = ['gender', 'stress_level', 'academic_work_impact']
for col in categorical_cols:
    df_processed[col] = le.fit_transform(df_processed[col])
    unique_vals = df[col].unique()
    encoded_vals = le.transform(unique_vals)
    mapping = dict(zip(unique_vals, encoded_vals))
    print(f"  Encoding '{col}': {mapping}")

# Feature Engineering: tambah fitur baru yang relevan
df_processed['social_to_total_ratio'] = (
    df_processed['social_media_hours'] / (df_processed['daily_screen_time_hours'] + 1e-5)
)
df_processed['gaming_to_total_ratio'] = (
    df_processed['gaming_hours'] / (df_processed['daily_screen_time_hours'] + 1e-5)
)
df_processed['notification_per_open'] = (
    df_processed['notifications_per_day'] / (df_processed['app_opens_per_day'] + 1e-5)
)
df_processed['sleep_deficit'] = 8 - df_processed['sleep_hours']  # Defisit tidur dari 8 jam ideal
df_processed['weekend_weekday_ratio'] = (
    df_processed['weekend_screen_time'] / (df_processed['daily_screen_time_hours'] + 1e-5)
)

print(f"\n✅ Feature Engineering: 5 fitur baru ditambahkan")
print(f"Total fitur akhir: {df_processed.shape[1] - 1}")  # -1 untuk target

In [ ]:
# Pisahkan fitur dan target
X = df_processed.drop('addicted_label', axis=1)
y = df_processed['addicted_label']

print(f"Fitur (X): {X.shape}")
print(f"Target (y): {y.shape}")
print(f"\nDaftar fitur ({len(X.columns)}):")
for i, col in enumerate(X.columns, 1):
    print(f"  {i:2d}. {col}")

In [ ]:
# Korelasi matriks
fig, ax = plt.subplots(figsize=(14, 11))
corr_matrix = df_processed.corr()
im = ax.imshow(corr_matrix, cmap='RdYlGn', vmin=-1, vmax=1, aspect='auto')
plt.colorbar(im, ax=ax, shrink=0.8)
ax.set_xticks(range(len(corr_matrix.columns)))
ax.set_yticks(range(len(corr_matrix.columns)))
ax.set_xticklabels(corr_matrix.columns, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(corr_matrix.columns, fontsize=8)

# Tambahkan nilai korelasi
for i in range(len(corr_matrix)):
    for j in range(len(corr_matrix.columns)):
        val = corr_matrix.iloc[i, j]
        color = 'white' if abs(val) > 0.6 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                fontsize=6, color=color)

ax.set_title('Gambar 2. Correlation Matrix Antar Fitur', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('gambar2_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Gambar 2 tersimpan: gambar2_correlation.png")

In [ ]:
# Oversampling (SMOTE manual) untuk menangani imbalanced data
# Karena library imbalanced-learn tidak tersedia, gunakan resample dari sklearn

X_train_raw, X_test, y_train_raw, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

# Gabungkan X_train dan y_train untuk oversampling
train_df = pd.concat([X_train_raw, y_train_raw], axis=1)

# Pisahkan kelas mayoritas dan minoritas
majority = train_df[train_df['addicted_label'] == 1]
minority = train_df[train_df['addicted_label'] == 0]

print(f"Sebelum balancing:")
print(f"  Kelas 1 (Kecanduan)        : {len(majority):,}")
print(f"  Kelas 0 (Tidak Kecanduan)  : {len(minority):,}")

# Upsample kelas minoritas
minority_upsampled = resample(
    minority, replace=True,
    n_samples=len(majority),
    random_state=RANDOM_STATE
)

train_balanced = pd.concat([majority, minority_upsampled])
X_train = train_balanced.drop('addicted_label', axis=1)
y_train = train_balanced['addicted_label']

print(f"\nSetelah balancing (Oversampling):")
print(f"  Kelas 1 (Kecanduan)        : {sum(y_train==1):,}")
print(f"  Kelas 0 (Tidak Kecanduan)  : {sum(y_train==0):,}")
print(f"\n✅ Data split: Train={len(X_train):,}, Test={len(X_test):,}")

# Standarisasi
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("✅ Standarisasi data selesai (StandardScaler)")

## 4. Pemodelan dan Hyperparameter Tuning

In [ ]:
# Definisi model dan grid hyperparameter
print("🔧 Konfigurasi Hyperparameter Tuning (GridSearchCV)")
print("=" * 60)

models_params = {
    'Gradient Boosting': {
        'model': GradientBoostingClassifier(random_state=RANDOM_STATE),
        'params': {
            'n_estimators': [100, 200],
            'max_depth': [3, 5],
            'learning_rate': [0.05, 0.1],
            'subsample': [0.8, 1.0],
        }
    },
    'Random Forest': {
        'model': RandomForestClassifier(random_state=RANDOM_STATE),
        'params': {
            'n_estimators': [100, 200],
            'max_depth': [None, 10, 20],
            'min_samples_split': [2, 5],
            'max_features': ['sqrt', 'log2'],
        }
    },
    'SVM': {
        'model': SVC(probability=True, random_state=RANDOM_STATE),
        'params': {
            'C': [0.1, 1, 10],
            'kernel': ['rbf', 'linear'],
            'gamma': ['scale', 'auto'],
        }
    },
    'Logistic Regression': {
        'model': LogisticRegression(random_state=RANDOM_STATE, max_iter=1000),
        'params': {
            'C': [0.01, 0.1, 1, 10],
            'penalty': ['l1', 'l2'],
            'solver': ['liblinear'],
        }
    }
}

for name, cfg in models_params.items():
    n_combinations = 1
    for v in cfg['params'].values():
        n_combinations *= len(v)
    print(f"  {name}: {n_combinations} kombinasi hyperparameter")

In [ ]:
# GridSearchCV dengan Stratified 5-Fold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

best_models = {}
best_params_all = {}

print("🔍 Menjalankan GridSearchCV...")
print("=" * 60)

for name, cfg in models_params.items():
    print(f"\n⏳ Tuning: {name}")
    
    # SVM dan LR membutuhkan data yang di-scale
    if name in ['SVM', 'Logistic Regression']:
        X_tr = X_train_scaled
        X_te = X_test_scaled
    else:
        X_tr = X_train.values
        X_te = X_test.values
    
    grid_search = GridSearchCV(
        estimator=cfg['model'],
        param_grid=cfg['params'],
        scoring='f1',
        cv=cv,
        n_jobs=-1,
        verbose=0
    )
    grid_search.fit(X_tr, y_train)
    
    best_models[name] = {
        'model': grid_search.best_estimator_,
        'X_test': X_te,
        'best_score_cv': grid_search.best_score_
    }
    best_params_all[name] = grid_search.best_params_
    
    print(f"  ✅ Best params : {grid_search.best_params_}")
    print(f"  📊 Best CV F1  : {grid_search.best_score_:.4f}")

print("\n✅ GridSearchCV selesai untuk semua model!")

## 5. Evaluasi Model

In [ ]:
# Evaluasi semua model
results = {}

print("📊 HASIL EVALUASI MODEL PADA DATA TEST")
print("=" * 70)

for name, cfg in best_models.items():
    model = cfg['model']
    X_te = cfg['X_test']
    
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc_score = roc_auc_score(y_test, y_prob)
    
    results[name] = {
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1,
        'AUC-ROC': auc_score,
        'y_pred': y_pred,
        'y_prob': y_prob
    }
    
    print(f"\n{'─'*50}")
    print(f"🤖 {name}")
    print(f"{'─'*50}")
    print(f"  Accuracy  : {acc:.4f} ({acc*100:.2f}%)")
    print(f"  Precision : {prec:.4f}")
    print(f"  Recall    : {rec:.4f}")
    print(f"  F1-Score  : {f1:.4f}")
    print(f"  AUC-ROC   : {auc_score:.4f}")

In [ ]:
# Tabel perbandingan lengkap
results_df = pd.DataFrame({
    name: {
        'Accuracy': f"{v['Accuracy']:.4f}",
        'Precision': f"{v['Precision']:.4f}",
        'Recall': f"{v['Recall']:.4f}",
        'F1-Score': f"{v['F1-Score']:.4f}",
        'AUC-ROC': f"{v['AUC-ROC']:.4f}",
    }
    for name, v in results.items()
}).T

print("\n📋 Tabel 1. Perbandingan Performa Model")
print(results_df.to_string())

In [ ]:
# Classification report model terbaik
best_model_name = max(results, key=lambda k: results[k]['F1-Score'])
print(f"🏆 Model Terbaik: {best_model_name}")
print(f"   F1-Score: {results[best_model_name]['F1-Score']:.4f}")
print(f"   AUC-ROC : {results[best_model_name]['AUC-ROC']:.4f}")
print()
print(f"Classification Report - {best_model_name}:")
print(classification_report(
    y_test,
    results[best_model_name]['y_pred'],
    target_names=['Tidak Kecanduan', 'Kecanduan']
))

In [ ]:
# Visualisasi: Confusion Matrix semua model
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
fig.suptitle('Gambar 3. Confusion Matrix Semua Model', fontsize=14, fontweight='bold')

axes_flat = axes.flatten()
colors_cm = ['Blues', 'Greens', 'Oranges', 'Purples']

for idx, (name, res) in enumerate(results.items()):
    cm = confusion_matrix(y_test, res['y_pred'])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                  display_labels=['Tidak Kecanduan', 'Kecanduan'])
    disp.plot(ax=axes_flat[idx], cmap=colors_cm[idx], colorbar=False)
    axes_flat[idx].set_title(
        f"{name}\nAcc={res['Accuracy']:.4f} | F1={res['F1-Score']:.4f}",
        fontweight='bold', fontsize=11
    )
    axes_flat[idx].set_xlabel('Predicted Label')
    axes_flat[idx].set_ylabel('True Label')

plt.tight_layout()
plt.savefig('gambar3_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Gambar 3 tersimpan: gambar3_confusion_matrix.png")

In [ ]:
# Visualisasi: ROC Curve semua model
fig, ax = plt.subplots(figsize=(9, 7))
colors_roc = ['#e74c3c', '#2ecc71', '#3498db', '#9b59b6']

for (name, res), color in zip(results.items(), colors_roc):
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, lw=2.5,
            label=f"{name} (AUC = {roc_auc:.4f})")

ax.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random Classifier (AUC = 0.50)')
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('Gambar 4. ROC Curve Perbandingan Model', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('gambar4_roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Gambar 4 tersimpan: gambar4_roc_curve.png")

In [ ]:
# Visualisasi: Perbandingan metrik semua model (bar chart)
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC']
model_names = list(results.keys())
x = np.arange(len(metrics))
width = 0.18
colors_bar = ['#e74c3c', '#2ecc71', '#3498db', '#9b59b6']

fig, ax = plt.subplots(figsize=(13, 7))

for i, (name, color) in enumerate(zip(model_names, colors_bar)):
    vals = [results[name][m] for m in metrics]
    bars = ax.bar(x + i * width, vals, width, label=name, color=color,
                  alpha=0.85, edgecolor='white')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                f'{val:.3f}', ha='center', va='bottom', fontsize=7.5, fontweight='bold')

ax.set_xlabel('Metrik Evaluasi', fontsize=12)
ax.set_ylabel('Nilai', fontsize=12)
ax.set_title('Gambar 5. Perbandingan Metrik Evaluasi Semua Model', fontsize=13, fontweight='bold')
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(metrics, fontsize=11)
ax.set_ylim([0.7, 1.05])
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('gambar5_perbandingan_metrik.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Gambar 5 tersimpan: gambar5_perbandingan_metrik.png")

## 6. Stratified K-Fold Cross Validation (10-Fold)

In [ ]:
# Stratified 10-Fold Cross Validation pada semua data
print("📊 STRATIFIED 10-FOLD CROSS VALIDATION")
print("=" * 60)

cv_10 = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)
cv_results = {}

for name, cfg in best_models.items():
    model = cfg['model']
    
    # Gunakan data yang sesuai
    if name in ['SVM', 'Logistic Regression']:
        X_cv = scaler.fit_transform(X)
    else:
        X_cv = X.values
    
    scores_acc = cross_val_score(model, X_cv, y, cv=cv_10, scoring='accuracy', n_jobs=-1)
    scores_f1  = cross_val_score(model, X_cv, y, cv=cv_10, scoring='f1', n_jobs=-1)
    scores_auc = cross_val_score(model, X_cv, y, cv=cv_10, scoring='roc_auc', n_jobs=-1)
    
    cv_results[name] = {
        'Accuracy':  scores_acc,
        'F1-Score':  scores_f1,
        'AUC-ROC':   scores_auc,
    }
    
    print(f"\n🤖 {name}")
    print(f"  Accuracy : {scores_acc.mean():.4f} ± {scores_acc.std():.4f}")
    print(f"  F1-Score : {scores_f1.mean():.4f} ± {scores_f1.std():.4f}")
    print(f"  AUC-ROC  : {scores_auc.mean():.4f} ± {scores_auc.std():.4f}")

print("\n✅ 10-Fold CV selesai!")

In [ ]:
# Visualisasi: Boxplot CV scores
fig, axes = plt.subplots(1, 3, figsize=(15, 6))
fig.suptitle('Gambar 6. Distribusi Skor 10-Fold Cross Validation', fontsize=13, fontweight='bold')

metrics_cv = ['Accuracy', 'F1-Score', 'AUC-ROC']
colors_box = ['#e74c3c', '#2ecc71', '#3498db', '#9b59b6']

for ax_idx, metric in enumerate(metrics_cv):
    data_box = [cv_results[name][metric] for name in model_names]
    bp = axes[ax_idx].boxplot(data_box, patch_artist=True, notch=False)
    
    for patch, color in zip(bp['boxes'], colors_box):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    
    axes[ax_idx].set_xticklabels(
        [n.replace(' ', '\n') for n in model_names], fontsize=9
    )
    axes[ax_idx].set_title(metric, fontweight='bold', fontsize=12)
    axes[ax_idx].set_ylabel('Score')
    axes[ax_idx].grid(axis='y', alpha=0.3)
    axes[ax_idx].set_ylim([0.75, 1.02])

plt.tight_layout()
plt.savefig('gambar6_cv_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Gambar 6 tersimpan: gambar6_cv_boxplot.png")

## 7. Feature Importance Analysis

In [ ]:
# Feature importance dari Gradient Boosting dan Random Forest
feature_names = list(X.columns)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
fig.suptitle('Gambar 7. Feature Importance Analysis', fontsize=13, fontweight='bold')

tree_models = ['Gradient Boosting', 'Random Forest']
colors_fi = ['#e74c3c', '#2ecc71']

for ax_idx, (name, color) in enumerate(zip(tree_models, colors_fi)):
    model = best_models[name]['model']
    importances = model.feature_importances_
    
    # Sort by importance
    indices = np.argsort(importances)[::-1]
    sorted_names = [feature_names[i] for i in indices]
    sorted_imps  = importances[indices]
    
    bars = axes[ax_idx].barh(
        range(len(sorted_names)), sorted_imps[::-1],
        color=color, alpha=0.8, edgecolor='white'
    )
    axes[ax_idx].set_yticks(range(len(sorted_names)))
    axes[ax_idx].set_yticklabels(sorted_names[::-1], fontsize=9)
    axes[ax_idx].set_xlabel('Importance Score', fontsize=11)
    axes[ax_idx].set_title(f'{name}\nFeature Importance', fontweight='bold', fontsize=11)
    axes[ax_idx].grid(axis='x', alpha=0.3)
    
    # Label nilai
    for bar, val in zip(bars, sorted_imps[::-1]):
        axes[ax_idx].text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
                          f'{val:.3f}', va='center', fontsize=8)

plt.tight_layout()
plt.savefig('gambar7_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Gambar 7 tersimpan: gambar7_feature_importance.png")

In [ ]:
# Top 5 fitur terpenting dari model terbaik
best_tree_model = best_models[best_model_name]['model']
importances = best_tree_model.feature_importances_
indices = np.argsort(importances)[::-1]

print(f"🏅 Top 10 Fitur Terpenting ({best_model_name}):")
print(f"{'Rank':<6} {'Fitur':<35} {'Importance':<12} {'Kumulatif'}")
print("-" * 65)
cumsum = 0
for rank, i in enumerate(indices[:10], 1):
    cumsum += importances[i]
    print(f"{rank:<6} {feature_names[i]:<35} {importances[i]:.4f}       {cumsum:.4f}")

## 8. Learning Curve Analysis

In [ ]:
# Learning Curve untuk model terbaik
print(f"📈 Menghitung Learning Curve: {best_model_name}...")

best_clf = best_models[best_model_name]['model']
if best_model_name in ['SVM', 'Logistic Regression']:
    X_lc = X_train_scaled
else:
    X_lc = X_train.values

train_sizes, train_scores, val_scores = learning_curve(
    best_clf, X_lc, y_train,
    train_sizes=np.linspace(0.1, 1.0, 10),
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    scoring='f1',
    n_jobs=-1
)

train_mean = train_scores.mean(axis=1)
train_std  = train_scores.std(axis=1)
val_mean   = val_scores.mean(axis=1)
val_std    = val_scores.std(axis=1)

fig, ax = plt.subplots(figsize=(9, 6))
ax.plot(train_sizes, train_mean, 'o-', color='#e74c3c', lw=2, label='Training Score')
ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std,
                alpha=0.15, color='#e74c3c')
ax.plot(train_sizes, val_mean, 'o-', color='#2ecc71', lw=2, label='Validation Score')
ax.fill_between(train_sizes, val_mean - val_std, val_mean + val_std,
                alpha=0.15, color='#2ecc71')

ax.set_xlabel('Ukuran Data Training', fontsize=12)
ax.set_ylabel('F1-Score', fontsize=12)
ax.set_title(f'Gambar 8. Learning Curve – {best_model_name}', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim([0.7, 1.05])

plt.tight_layout()
plt.savefig('gambar8_learning_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Gambar 8 tersimpan: gambar8_learning_curve.png")

## 9. Ringkasan Hasil untuk Jurnal

In [ ]:
print("=" * 70)
print("📄 RINGKASAN HASIL PENELITIAN")
print("=" * 70)

print("\n🗂️  DATASET")
print(f"  Jumlah sampel        : 7.500")
print(f"  Jumlah fitur asli    : 13")
print(f"  Fitur setelah FE     : {len(X.columns)}")
print(f"  Kelas (binary)       : Kecanduan (1) / Tidak Kecanduan (0)")
print(f"  Metode balancing     : Oversampling (Majority Class Resample)")

print("\n⚙️  METODOLOGI")
print(f"  Preprocessing        : Label Encoding, Feature Engineering, StandardScaler")
print(f"  Split data           : 80% Train / 20% Test (Stratified)")
print(f"  Tuning               : GridSearchCV, Stratified 5-Fold CV")
print(f"  Evaluasi             : Stratified 10-Fold Cross Validation")

print("\n🏆 PERBANDINGAN MODEL (Test Set)")
print(f"  {'Model':<22} {'Acc':>8} {'Prec':>8} {'Rec':>8} {'F1':>8} {'AUC':>8}")
print("  " + "-" * 65)
for name, res in results.items():
    marker = " ← BEST" if name == best_model_name else ""
    print(f"  {name:<22} {res['Accuracy']:>8.4f} {res['Precision']:>8.4f} "
          f"{res['Recall']:>8.4f} {res['F1-Score']:>8.4f} {res['AUC-ROC']:>8.4f}{marker}")

print("\n📊 10-FOLD CROSS VALIDATION (Mean ± Std)")
print(f"  {'Model':<22} {'Acc Mean±Std':>18} {'F1 Mean±Std':>18} {'AUC Mean±Std':>18}")
print("  " + "-" * 78)
for name, cvr in cv_results.items():
    print(f"  {name:<22} "
          f"{cvr['Accuracy'].mean():.4f}±{cvr['Accuracy'].std():.4f}   "
          f"{cvr['F1-Score'].mean():.4f}±{cvr['F1-Score'].std():.4f}   "
          f"{cvr['AUC-ROC'].mean():.4f}±{cvr['AUC-ROC'].std():.4f}")

print("\n🖼️  GAMBAR YANG DIHASILKAN")
gambar_list = [
    ("gambar1_eda.png",              "Eksplorasi Data Awal (EDA)"),
    ("gambar2_correlation.png",      "Correlation Matrix"),
    ("gambar3_confusion_matrix.png", "Confusion Matrix (4 Model)"),
    ("gambar4_roc_curve.png",        "ROC Curve Perbandingan"),
    ("gambar5_perbandingan_metrik.png", "Bar Chart Metrik Evaluasi"),
    ("gambar6_cv_boxplot.png",       "Boxplot 10-Fold CV"),
    ("gambar7_feature_importance.png", "Feature Importance"),
    ("gambar8_learning_curve.png",   "Learning Curve"),
]
for fname, desc in gambar_list:
    print(f"  ✅ {fname:<40} → {desc}")

print("\n" + "=" * 70)
print("✅ Notebook selesai dijalankan. Siap untuk penulisan jurnal Sinta 3!")
print("=" * 70)